In [1]:
!pip install eyepop==3.12.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.5 MB/s eta 0:00:00
  Attempting uninstall: cryptography
    Found existing installation: cryptography 50.0.0
    Uninstalling cryptography-50.0.0:
      Successfully uninstalled cryptography-50.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyopenssl 26.4.0 requires cryptography<51,>=49.0.0, but you have cryptography 46.0.7 which is incompatible.


In [2]:
import getpass

EYEPOP_ACCOUNT_ID=input("Enter your Account UUID: ")
EYEPOP_API_KEY=getpass.getpass('Enter your API KEY: ')

Enter your Account UUID: a5184defa8e847248f589d35080efbfa
Enter your API KEY: ··········


In [3]:
NAMESPACE_PREFIX="datasciencealliance-org" # Add your namespace-prefix here

# Define Ability

In [4]:
from eyepop import EyePopSdk
from eyepop.data.data_types import InferRuntimeConfig, VlmAbilityGroupCreate, VlmAbilityCreate, TransformInto
from eyepop.worker.worker_types import CropForward, ForwardComponent, FullForward, InferenceComponent, Pop
import json


LIVESTOCK_HEADCOUNT_HEALTHFLAGGING_DESCRIPTION_PROMPT = """
Analyze the entire video and identify all visible livestock. Count distinct animals across the video and flag individuals that are visibly injured, lame, or isolated.

Count each animal only once, even if it appears across multiple frames. Identify the livestock species and report the number of each species. "species_count" is a dictionary where each livestock species is a key and its corresponding number of visible animals is the value.

Health flags:
"injured": The animal has a clearly visible physical injury, such as a wound, bleeding, swelling, bandage, deformity, or visibly injured limb. Only report injuries that are visually observable. Do not infer internal injuries or disease.

"lame": The animal shows a clearly abnormal gait while moving, such as repeated limping, uneven walking, or consistently avoiding weight on a limb. Determine lameness from movement across multiple frames, not from a single frame or static posture. Only report lameness when the abnormal movement is clearly visible.

"isolated": The animal is clearly separated from the main group in the current frame, with a noticeable area of empty space between it and the other animals. Flag isolation only when one animal is visibly apart from a clearly grouped majority. Do not require evidence that the separation persists across the video.

For each flagged animal, provide an "animal_reference" that helps a reviewer distinguish that individual from the other animals using visible characteristics such as species, coat color or markings, and relative location. For example: "brown goat with white markings on the left side of the pen", "black cow separated from the group on the far right", or "white sheep near the fence". Do not assign arbitrary identifiers such as "goat 1" or invent characteristics that are not visible.

An animal may have more than one flag. If the same animal is both injured and lame, for example, include both flags in the same entry.

Only report a health flag when there is clear visual evidence. If a condition cannot be determined confidently from the video, do not flag it. Do not diagnose diseases or infer the cause of an injury or abnormal behavior.

Return only valid JSON in exactly this structure:

{
  "total_headcount": null,
  "species_count": null,
  "flagged_animals": [
    {
      "species": null,
      "animal_reference": null,
      "flags": null,
      "description": null
    }
  ]
}

If no animals are flagged, return an empty "flagged_animals" array.
"""


ability_prototypes = [
    VlmAbilityCreate(
        name=f"{NAMESPACE_PREFIX}.describe.livestock-headcount-healthflagging",
        description="Counts livestock and flags visibly injured, lame, or isolated animals in video footage.",
        worker_release="qwen3-instruct",
        text_prompt=LIVESTOCK_HEADCOUNT_HEALTHFLAGGING_DESCRIPTION_PROMPT,
        transform_into=TransformInto(),
        config=InferRuntimeConfig(
            max_new_tokens=600,
            fps = 5,
            image_size=640
        ),
        is_public=False
    )
]

# Create Ability

In [5]:
with EyePopSdk.dataEndpoint(api_key=EYEPOP_API_KEY, account_id=EYEPOP_ACCOUNT_ID) as endpoint:
    for ability_prototype in ability_prototypes:
        ability_group = endpoint.create_vlm_ability_group(VlmAbilityGroupCreate(
            name=ability_prototype.name,
            description=ability_prototype.description,
            default_alias_name=ability_prototype.name,
        ))
        ability = endpoint.create_vlm_ability(
            create=ability_prototype,
            vlm_ability_group_uuid=ability_group.uuid,
        )
        ability = endpoint.publish_vlm_ability(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
        )
        ability = endpoint.add_vlm_ability_alias(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
            tag_name="latest"
        )
        print(f"created ability {ability.uuid} with alias entries {ability.alias_entries}")

created ability 06a91ce1d4bc76918000f2006138d1c1 with alias entries [AbilityAliasEntry(alias='datasciencealliance-org.describe.livestock-headcount-healthflagging', tag='1.0.17'), AbilityAliasEntry(alias='datasciencealliance-org.describe.livestock-headcount-healthflagging', tag='latest')]


# Evaluate on a single video

In [6]:
from pathlib import Path
import json

pop = Pop(components=[
    InferenceComponent(
        ability=f"{NAMESPACE_PREFIX}.describe.livestock-headcount-healthflagging:latest",
        targetFps= "1"
    )
])

with EyePopSdk.workerEndpoint(api_key=EYEPOP_API_KEY) as endpoint:
    endpoint.set_pop(pop)

    sample_vid_path = Path("/content/livestock11.mp4")

    job = endpoint.upload(sample_vid_path)
    i = 1
    prev_text = ""
    while result := job.predict():
        text = result["texts"][0]["text"]
        if text==prev_text:
           continue
        print(f"Result {i}")
        print("-"*100)
        i+=1
        prev_text = text

        # Remove ```json and ```
        text = text.replace("```json", "").replace("```", "").strip()

        # Parse JSON
        data = json.loads(text)

        # Pretty print
        print(json.dumps(data, indent=2))










Result 1
----------------------------------------------------------------------------------------------------
{
  "total_headcount": 8,
  "species_count": {
    "goat": 8
  },
  "flagged_animals": []
}
Result 2
----------------------------------------------------------------------------------------------------
{
  "total_headcount": 7,
  "species_count": {
    "goat": 7
  },
  "flagged_animals": []
}
Result 3
----------------------------------------------------------------------------------------------------
{
  "total_headcount": 8,
  "species_count": {
    "goat": 8
  },
  "flagged_animals": []
}
Result 4
----------------------------------------------------------------------------------------------------
{
  "total_headcount": 7,
  "species_count": {
    "goat": 7
  },
  "flagged_animals": [
    {
      "species": "goat",
      "animal_reference": "white goat with pinkish bruising on its hindquarters lying on the ground in the foreground",
      "flags": [
        "injured"
      ],
